# imports and preprocessing

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import LabelEncoder
import psutil, os

# Load & preprocess data
df = pd.read_csv("CongressionalVotingID_cleaned.csv")
df = df.drop(columns=["ID"])
df["class"] = LabelEncoder().fit_transform(df["class"])

X = df.drop(columns=["class"])
y = df["class"]

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Convert to tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)

# Model Definition

In [17]:
class VotingClassifier(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(VotingClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.Sigmoid()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

model = VotingClassifier(input_size=X_train.shape[1], hidden_size=16, output_size=2)



# Training Loop

In [18]:
#Loss & optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

#Train
epochs = 50
for epoch in range(epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


Epoch 10/50 - Loss: 0.5056
Epoch 20/50 - Loss: 0.3054
Epoch 30/50 - Loss: 0.2225
Epoch 40/50 - Loss: 0.1902
Epoch 50/50 - Loss: 0.1617


# Evaluation:

In [ ]:
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\nClassification Report:\n")
    print(classification_report(y_test_tensor, predicted))

# Stats
param_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
ram_used = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"\nLearnable Parameters: {param_count}")
print(f"Virtual RAM Used: {ram_used:.2f} MB")


Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95        21
           1       1.00      0.91      0.95        23

    accuracy                           0.95        44
   macro avg       0.96      0.96      0.95        44
weighted avg       0.96      0.95      0.95        44


Learnable Parameters: 290
Virtual RAM Used: 93.39 MB


## Model B – ReLU Activation, 32 Hidden Units

In this variation, we modify the baseline neural network by:
- Increasing the hidden layer size from 16 to 32 units
- Replacing the Sigmoid activation with ReLU

The rest of the model structure, training parameters (50 epochs, learning rate 0.01), and evaluation setup remain the same. This helps us observe the impact of activation function and hidden layer size on performance and resource usage.


In [ ]:
class VotingClassifier_B(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(VotingClassifier_B, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

# Model B
model_b = VotingClassifier_B(input_size=X_train.shape[1], hidden_size=32, output_size=2)


In [21]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_b.parameters(), lr=0.01)

epochs = 50
for epoch in range(epochs):
    model_b.train()
    outputs = model_b(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model B] Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


[Model B] Epoch 10/50 - Loss: 0.2492
[Model B] Epoch 20/50 - Loss: 0.1550
[Model B] Epoch 30/50 - Loss: 0.0926
[Model B] Epoch 40/50 - Loss: 0.0603
[Model B] Epoch 50/50 - Loss: 0.0413


In [22]:
model_b.eval()
with torch.no_grad():
    predictions = model_b(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model B] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

# Parameters and RAM
param_count_b = sum(p.numel() for p in model_b.parameters() if p.requires_grad)
ram_used_b = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model B] Learnable Parameters: {param_count_b}")
print(f"[Model B] Virtual RAM Used: {ram_used_b:.2f} MB")



[Model B] Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95        21
           1       1.00      0.91      0.95        23

    accuracy                           0.95        44
   macro avg       0.96      0.96      0.95        44
weighted avg       0.96      0.95      0.95        44

[Model B] Learnable Parameters: 578
[Model B] Virtual RAM Used: 95.61 MB


##  Model C – 2 Hidden Layers (32 → 16), Tanh Activation

In this configuration, we extend the baseline network by:
- Adding a second hidden layer
- Using 32 units in the first layer and 16 in the second
- Applying the Tanh activation function between both layers

The goal is to see how a deeper architecture and different activation affect performance, learning behavior, and resource usage.


In [23]:
class VotingClassifier_C(nn.Module):
    def __init__(self, input_size, hidden1_size, hidden2_size, output_size):
        super(VotingClassifier_C, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden1_size)
        self.act1 = nn.Tanh()
        self.fc2 = nn.Linear(hidden1_size, hidden2_size)
        self.act2 = nn.Tanh()
        self.fc3 = nn.Linear(hidden2_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act1(x)
        x = self.fc2(x)
        x = self.act2(x)
        x = self.fc3(x)
        return x

# Initialize Model C
model_c = VotingClassifier_C(
    input_size=X_train.shape[1], hidden1_size=32, hidden2_size=16, output_size=2
)


In [24]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_c.parameters(), lr=0.01)

epochs = 50
for epoch in range(epochs):
    model_c.train()
    outputs = model_c(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model C] Epoch {epoch+1}/{epochs} - Loss: {loss.item():.4f}")


[Model C] Epoch 10/50 - Loss: 0.2282
[Model C] Epoch 20/50 - Loss: 0.1152
[Model C] Epoch 30/50 - Loss: 0.0548
[Model C] Epoch 40/50 - Loss: 0.0260
[Model C] Epoch 50/50 - Loss: 0.0109


In [25]:
model_c.eval()
with torch.no_grad():
    predictions = model_c(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model C] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

# Parameters and RAM
param_count_c = sum(p.numel() for p in model_c.parameters() if p.requires_grad)
ram_used_c = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model C] Learnable Parameters: {param_count_c}")
print(f"[Model C] Virtual RAM Used: {ram_used_c:.2f} MB")



[Model C] Classification Report:

              precision    recall  f1-score   support

           0       0.91      1.00      0.95        21
           1       1.00      0.91      0.95        23

    accuracy                           0.95        44
   macro avg       0.96      0.96      0.95        44
weighted avg       0.96      0.95      0.95        44

[Model C] Learnable Parameters: 1074
[Model C] Virtual RAM Used: 97.25 MB


##  Model Comparison – Voting Dataset 

| Model | Hidden Layers | Hidden Units  | Activation | Accuracy | Final Loss | Parameters | RAM Used (MB) |
|-------|----------------|----------------|------------|----------|-------------|------------|----------------|
| A     | 1              | 16             | Sigmoid    | 95%      | 0.2093      | 290        | 163.23         |
| B     | 1              | 32             | ReLU       | 95%      | 0.0374      | 578        | 163.55         |
| C     | 2              | 32 → 16        | Tanh       | 95%      | 0.0072      | 1,074      | 160.69         |

**Notes:**
- All models performed equally well in classification metrics.
- Deeper or wider networks reduced the training loss faster and more effectively.
- Resource usage increased slightly with model complexity but stayed efficient overall.


## Model D – 1 Hidden Layer (Heuristic: 10 Units), ReLU Activation

This model applies a widely accepted heuristic for determining the number of hidden neurons:
- Hidden size ≈ 2/3 of input size → 10 neurons
- Activation function: ReLU
- Structure: 1 hidden layer
- Other training settings (optimizer, loss, epochs) remain the same

We use this model to evaluate whether heuristic-based design offers a strong trade-off between performance and efficiency.


In [26]:
class VotingClassifier_D(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(VotingClassifier_D, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.activation = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)
        return x

model_d = VotingClassifier_D(input_size=15, hidden_size=10, output_size=2)


In [27]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model_d.parameters(), lr=0.01)

for epoch in range(50):
    model_d.train()
    outputs = model_d(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f"[Model D] Epoch {epoch+1}/50 - Loss: {loss.item():.4f}")


[Model D] Epoch 10/50 - Loss: 0.4197
[Model D] Epoch 20/50 - Loss: 0.2240
[Model D] Epoch 30/50 - Loss: 0.1729
[Model D] Epoch 40/50 - Loss: 0.1317
[Model D] Epoch 50/50 - Loss: 0.0935


In [28]:
model_d.eval()
with torch.no_grad():
    predictions = model_d(X_test_tensor)
    _, predicted = torch.max(predictions, 1)
    print("\n[Model D] Classification Report:\n")
    print(classification_report(y_test_tensor, predicted))

param_count_d = sum(p.numel() for p in model_d.parameters() if p.requires_grad)
ram_used_d = psutil.Process(os.getpid()).memory_info().rss / 1024**2
print(f"[Model D] Learnable Parameters: {param_count_d}")
print(f"[Model D] Virtual RAM Used: {ram_used_d:.2f} MB")



[Model D] Classification Report:

              precision    recall  f1-score   support

           0       0.95      1.00      0.98        21
           1       1.00      0.96      0.98        23

    accuracy                           0.98        44
   macro avg       0.98      0.98      0.98        44
weighted avg       0.98      0.98      0.98        44

[Model D] Learnable Parameters: 182
[Model D] Virtual RAM Used: 99.66 MB


## Model E – Grid Search Over Layers and Activations (Voting Dataset)

We perform a grid search to find the best architecture for the Voting dataset by varying:
- The number and size of hidden layers (1-layer and 2-layer models)
- The activation function (`ReLU`, `Sigmoid`, `Tanh`)

Each model is trained for 50 epochs using the Adam optimizer and CrossEntropyLoss. Performance is evaluated using **macro-averaged F1-score**, and we also track final loss, parameter count, and RAM usage.


In [31]:
from sklearn.metrics import f1_score

# Activation functions
activation_map = {
    "relu": nn.ReLU(),
    "sigmoid": nn.Sigmoid(),
    "tanh": nn.Tanh()
}

# Grid of layer configs and activations
layer_configs = [
    [8], [12], [16],
    [16, 8], [24, 12], [32, 16]
]
activations = ["relu", "sigmoid", "tanh"]
results_voting = []

# Loop through combinations
for layers in layer_configs:
    for act_name in activations:
        act_fn = activation_map[act_name]

        class VotingModelE(nn.Module):
            def __init__(self):
                super(VotingModelE, self).__init__()
                self.layers = nn.ModuleList()
                prev_size = X_train_tensor.shape[1]
                for h in layers:
                    self.layers.append(nn.Linear(prev_size, h))
                    prev_size = h
                self.output = nn.Linear(prev_size, 2)
                self.act = act_fn

            def forward(self, x):
                for layer in self.layers:
                    x = self.act(layer(x))
                return self.output(x)

        model_e = VotingModelE()
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model_e.parameters(), lr=0.01)

        for epoch in range(50):
            model_e.train()
            outputs = model_e(X_train_tensor)
            loss = criterion(outputs, y_train_tensor)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

        # Evaluation
        model_e.eval()
        with torch.no_grad():
            preds = model_e(X_test_tensor)
            _, predicted = torch.max(preds, 1)
            f1 = f1_score(y_test_tensor, predicted, average="macro")
            acc = (predicted == y_test_tensor).sum().item() / len(y_test_tensor)
            final_loss = loss.item()
            params = sum(p.numel() for p in model_e.parameters() if p.requires_grad)
            ram = psutil.Process(os.getpid()).memory_info().rss / 1024**2

            results_voting.append({
                "layers": layers,
                "activation": act_name,
                "f1": round(f1, 4),
                "accuracy": round(acc, 4),
                "loss": round(final_loss, 4),
                "params": params,
                "ram": round(ram, 2)
            })

# Sort and display
df_grid_voting = pd.DataFrame(sorted(results_voting, key=lambda x: x["f1"], reverse=True))
df_grid_voting


,layers,activation,f1,accuracy,loss,params,ram
0,[8],relu,0.9773,0.9773,0.2240,146,105.20
1,[12],relu,0.9773,0.9773,0.0898,218,105.45
2,[16],relu,0.9773,0.9773,0.0583,290,105.58
3,[16],tanh,0.9773,0.9773,0.0798,290,105.58
4,"[16, 8]",tanh,0.9773,0.9773,0.0373,410,105.81
5,"[24, 12]",relu,0.9773,0.9773,0.0339,710,105.86
6,[8],sigmoid,0.9545,0.9545,0.3120,146,105.30
7,[8],tanh,0.9545,0.9545,0.1431,146,105.42
8,[12],sigmoid,0.9545,0.9545,0.1928,218,105.52
9,[12],tanh,0.9545,0.9545,0.1041,218,105.56


## Model E – Grid Search Optimized Architecture (Voting Dataset)

We conducted a grid search over:
- 6 hidden layer configurations (1-layer and 2-layer networks)
- 3 activation functions: ReLU, Sigmoid, Tanh

Each model was trained for 50 epochs and evaluated using **macro-averaged F1-score**.  
The best configuration was:
- **2 hidden layers**: 24 → 12
- **ReLU activation**
- F1-score and accuracy: 97.73%
- Final loss: 0.0339
- Parameters: 710
- RAM usage: ~106 MB

This model offered a balance of high performance and moderate complexity.


## Model Comparison – Voting Dataset (Models A to E)

| Model | Hidden Layers | Hidden Units   | Activation | Accuracy | F1-score | Final Loss | Parameters | RAM Used (MB) |
|-------|----------------|----------------|------------|----------|----------|-------------|------------|----------------|
| A     | 1              | 16             | Sigmoid    | 95%      | 0.95     | 0.1617      | 290        | 93.39          |
| B     | 1              | 32             | ReLU       | 95%      | 0.95     | 0.0413      | 578        | 95.61          |
| C     | 2              | 32 → 16        | Tanh       | 95%      | 0.95     | 0.0109      | 1074       | 97.25          |
| D     | 1              | 10             | ReLU       | 98%      | 0.98     | 0.0935      | 182        | 99.66          |
| E     | 2              | 24 → 12        | ReLU       | 97.73%   | 0.9773   | 0.0339      | 710        | 105.86         |
